# Eksik Gözlem Rassallık Testi (Little's MCAR Test)

Bir önceki konuda (Eksik Gözlem Nasıl Tespit Edilir?), `msno.heatmap()` 
ile eksikliklerin **görsel olarak** ilişkili olup olmadığına bakmıştık. 
Bu konuda, aynı soruyu **resmi bir hipotez testiyle** cevaplıyoruz: 
**Little's MCAR Testi.**

## MCAR/MAR/MNAR Ayrımını Hatırlayalım
- **MCAR (Missing Completely At Random):** Eksiklik tamamen rastgele, hiçbir değişkenle ilişkili değil
- **MAR (Missing At Random):** Eksiklik, gözlenen başka bir değişkenle ilişkili
- **MNAR (Missing Not At Random):** Eksiklik, değerin kendisiyle ilişkili (en tehlikeli, tespit etmesi en zor)

## Little's MCAR Test Ne Yapar?
Bu test, **sadece MCAR olup olmadığını** test eder — "eksiklik gerçekten TAMAMEN rastgele mi?" sorusuna cevap verir.

## Hipotezler
- **H0:** Veri MCAR'dır (eksiklik tamamen rastgele)
- **H1:** Veri MCAR değildir (eksiklik bir örüntü/ilişki içeriyor — MAR veya MNAR olabilir, test bu ikisini ayırt etmez)

## Karar Kuralı
- **p < α (0.05)** → H0 reddedilir → Veri **MCAR DEĞİL** → dikkatli 
  olunmalı, basit silme (listwise deletion) yanlış sonuçlara yol açabilir
- **p ≥ α (0.05)** → H0 reddedilemez → Veri **MCAR olabilir** → basit 
  yöntemler (satır silme gibi) nispeten güvenli

## Önemli Sınırlama
Little's Test, **sadece MCAR vs "MCAR değil"** ayrımını yapar — "MCAR 
değil" çıkarsa, bunun MAR mı MNAR mı olduğunu **söylemez.** Bu ayrımı 
yapmak için domain bilgisi ve görsel inceleme birlikte kullanılır.

## Python'da Neden R Kullandık?
Little's MCAR Testi için Python'da yaygın/stabil bir kütüphane 
bulunamadı (XeroGraph denendi, ancak ağır ve çakışan bağımlılıklar 
— eski sürüm sabitlemeleri, torch gibi devasa paketler — nedeniyle 
kurulumu bu ortamda başarısız oldu). Bu test, R'da `naniar` paketinin 
`mcar_test()` fonksiyonuyla çok daha stabil ve kolay çalıştırılabiliyor. 
Bu nedenle analiz R'da yapılmış, kod ve sonuç aşağıda paylaşılmıştır 
(R script dosyası: `R/MCAR_test.R`).

## R Kodu
```r
library(naniar)

df <- data.frame(
  Yas = rnorm(200, 35, 10),
  Gelir = rnorm(200, 5000, 1500),
  Telefon = sample(c("555-1234", "555-5678", NA), 200, replace=TRUE, prob=c(0.4,0.4,0.2)),
  Memnuniyet = rnorm(200, 70, 15)
)

mcar_test(df)
```

## Sonuç

R'da çalıştırılan Little's MCAR Testi sonucuna göre:

| statistic | df | p.value | missing.patterns |
|---|---|---|---|
| 5.09 | 3 | 0.165 | 2 |

p-değeri (0.165), α=0.05'ten büyük olduğu için **H0 reddedilemez** — bu 
veri setinde eksikliğin **MCAR (tamamen rastgele) olabileceği** 
sonucuna varılmıştır. Veri setinde 2 farklı eksiklik örüntüsü 
(missing.patterns) tespit edilmiştir.

Bu sonuç, bir önceki konudaki Python analizinden (msno.heatmap ile 
Telefon-Email arasında 0.5 korelasyon bulunan) **farklı bir veri seti** 
kullanıldığı için tutarlıdır — o veride Telefon ve Email kasıtlı olarak 
birbirine bağımlı kurgulanmıştı, bu örnekte ise böyle bir kasıtlı 
ilişki bulunmadığından, MCAR testi de "rastgele olabilir" sonucuna 
varmıştır. Bu karşılaştırma, aynı zamanda **eksiklik gerçekten 
ilişkiliyse Little's Test'in bunu p<0.05 ile yakalayabileceğini**, 
ilişkisizse (bizim bu örneğimizde olduğu gibi) p>0.05 ile "MCAR" 
sonucuna varabileceğini göstermektedir.